# Qwen3.5-9B Grading QLoRA — Colab vLLM Server

This notebook is **inference/deployment only**. It contains no dataset loading, QLoRA/SFT training, validation, checkpointing, or evaluation code.

It serves:

- Base model: `Qwen/Qwen3.5-9B`
- LoRA adapter: `SmuFypTeam5/GradingQlora`
- API model name: `grading`
- API: OpenAI-compatible vLLM server
- External access: Cloudflare Quick Tunnel, so an app running in local Docker can call the Colab GPU

### Colab setup

1. Open **Runtime → Change runtime type → GPU**.
2. An L4/A100 is recommended. The notebook enables vLLM BitsAndBytes 4-bit loading by default to reduce VRAM usage.
3. Make the sibling `.env` file available in the notebook's working folder in Colab (normally `/content`). It can contain `HF_TOKEN` and `VLLM_API_KEY`.
4. Alternatively, add `HF_TOKEN` and `VLLM_API_KEY` in **Colab Secrets**. If a required value is missing, the notebook asks for it securely.
5. Run cells from top to bottom. When prompted, authorize the Drive mount so the model and adapter can be saved under `MyDrive/qlora-vllm-models` before vLLM starts.

The Cloudflare Quick Tunnel URL changes when the Colab runtime/tunnel restarts.

## 1. Check the Colab GPU

In [1]:
!nvidia-smi

Sat Sep 26 09:35:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install serving dependencies

This installs only inference/serving packages. There is no TRL, PEFT training setup, dataset package, or trainer configuration here.

If Colab asks you to restart the runtime after installation, restart it and continue from the next cell.

In [2]:
%pip install -q -U vllm==0.30.0 vllm-bnb-plugin==0.0.3 openai huggingface_hub requests python-dotenv

# Cloudflare tunnel binary used to expose vLLM to your local Docker app.
!wget -q -O /usr/local/bin/cloudflared \
  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

# This notebook serves a text-only grading model. Colab may ship a TorchAudio
# wheel built for a different CUDA version than its current PyTorch. Transformers
# probes TorchAudio during vLLM startup, so remove the unused incompatible wheel.
%pip uninstall -y torchaudio

# vLLM 0.29+ provides BitsAndBytes through the separate vllm-bnb-plugin.
# Check its registration as well as the CLI before downloading model weights.
import subprocess
import sys
import torch
import bitsandbytes
from importlib.metadata import version

subprocess.run(["vllm", "--help"], check=True, stdout=subprocess.DEVNULL)
subprocess.run(
    [
        sys.executable,
        "-c",
        "from vllm.plugins import load_general_plugins; "
        "load_general_plugins(); "
        "from vllm.model_executor.layers.quantization import get_quantization_config; "
        "get_quantization_config('bitsandbytes')",
    ],
    check=True,
)
print(f"Dependency check passed: PyTorch {torch.__version__} (CUDA {torch.version.cuda}), BitsAndBytes {bitsandbytes.__version__}, vLLM {version('vllm')} with BitsAndBytes quantization.")

Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0
Dependency check passed: PyTorch 2.13.0+cu130 (CUDA 13.0), BitsAndBytes 0.50.2, vLLM 0.30.0 with BitsAndBytes quantization.


## 3. Configuration and secrets

In [3]:
import os
import getpass
import subprocess
import time
import re
from pathlib import Path

import requests
import torch
from huggingface_hub import login
from dotenv import load_dotenv

# In notebooks, the working directory is the practical equivalent of the
# notebook's folder. In Colab, place the sibling .env file in /content.
ENV_PATH = Path.cwd() / ".env"
load_dotenv(dotenv_path=ENV_PATH, override=False)
print(f"Loaded .env from {ENV_PATH}" if ENV_PATH.is_file() else f"No .env found at {ENV_PATH}; using environment/Colab Secrets.")

BASE_MODEL = "Qwen/Qwen3.5-9B"
LORA_REPO = "SmuFypTeam5/GradingQlora"
SERVED_MODEL_NAME = "grading"

PORT = 8000
MAX_MODEL_LEN = 4096
MAX_NUM_SEQS = 4
GPU_MEMORY_UTILIZATION = 0.90
MAX_LORA_RANK = 16
ENFORCE_EAGER = True  # Match serve_qwen35_vllm.sh; skip CUDA graph capture

# Recommended for typical Colab GPUs because the unquantized model is large.
# On an A100 40/80GB you can set this to False if you prefer full BF16/FP16 weights.
USE_BITSANDBYTES_4BIT = True


def get_secret(name, required=True):
    value = os.environ.get(name)
    if value:
        return value

    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None

    if not value and required:
        value = getpass.getpass(f"Enter {name}: ")

    return value


HF_TOKEN = get_secret("HF_TOKEN", required=True)
VLLM_API_KEY = get_secret("VLLM_API_KEY", required=True)


if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. In Colab, select Runtime → Change runtime type → GPU.")

gpu = torch.cuda.get_device_properties(0)
# T4 (compute capability 7.5) has no native BF16 even if PyTorch reports emulation.
VLLM_DTYPE = "bfloat16" if gpu.major >= 8 and torch.cuda.is_bf16_supported() else "float16"
print(f"GPU: {gpu.name}")
print(f"VRAM: {gpu.total_memory / 1024**3:.1f} GiB")
print("Base model:", BASE_MODEL)
print("LoRA adapter:", LORA_REPO)
print("Served LoRA name:", SERVED_MODEL_NAME)
print("4-bit BitsAndBytes:", USE_BITSANDBYTES_4BIT)
print("Max sequences:", MAX_NUM_SEQS)
print("Eager mode (CUDA graphs disabled):", ENFORCE_EAGER)

No .env found at /content/.env; using environment/Colab Secrets.
GPU: Tesla T4
VRAM: 14.6 GiB
Base model: Qwen/Qwen3.5-9B
LoRA adapter: SmuFypTeam5/GradingQlora
Served LoRA name: grading
4-bit BitsAndBytes: True


## 4. Save model files to Google Drive

Mount Google Drive, then save the base model and LoRA adapter under `MyDrive/qlora-vllm-models`. On the first run, this reuses any complete Hugging Face cache already in the runtime and copies it to Drive. On later runtimes, it restores the Drive files to local Colab storage before vLLM starts. Allow about 20 GB of Drive space for the base model.


In [ ]:
import shutil
from huggingface_hub import snapshot_download
from huggingface_hub.constants import HF_HUB_CACHE
from google.colab import drive

drive.mount("/content/drive")
DRIVE_MODELS_DIR = Path("/content/drive/MyDrive/qlora-vllm-models")
LOCAL_MODELS_DIR = Path("/content/qlora-vllm-models")
DRIVE_MODELS_DIR.mkdir(parents=True, exist_ok=True)


def stage_model(repo_id, folder_name, required_file):
    drive_dir = DRIVE_MODELS_DIR / folder_name
    drive_marker = drive_dir / ".snapshot_commit"

    if not drive_marker.is_file():
        # snapshot_download uses the local Hub cache, including files from a
        # previous attempt in this runtime, before fetching anything missing.
        snapshot = Path(snapshot_download(repo_id=repo_id, token=HF_TOKEN or None))
        print(f"Saving {repo_id} to Google Drive. This may take a while.")
        shutil.copytree(snapshot, drive_dir, dirs_exist_ok=True)
        drive_marker.write_text(snapshot.name)
        print(f"Saved {repo_id} to {drive_dir}")
        return str(snapshot)

    revision = drive_marker.read_text().strip()
    if not revision or not (drive_dir / required_file).is_file():
        raise RuntimeError(f"Incomplete Drive copy at {drive_dir}. Remove it and rerun this cell.")

    # Reuse the runtime's local Hub cache when it is already complete.
    cached = Path(HF_HUB_CACHE) / f"models--{repo_id.replace('/', '--')}" / "snapshots" / revision
    if cached.is_dir() and (cached / required_file).is_file():
        print(f"Using local cache for {repo_id}")
        return str(cached)

    local_dir = LOCAL_MODELS_DIR / folder_name
    local_marker = local_dir / ".snapshot_commit"
    if not local_marker.is_file() or local_marker.read_text().strip() != revision:
        print(f"Restoring {repo_id} from Google Drive to local Colab storage.")
        shutil.copytree(
            drive_dir,
            local_dir,
            dirs_exist_ok=True,
            ignore=shutil.ignore_patterns(".snapshot_commit"),
        )
        local_marker.write_text(revision)
    return str(local_dir)


BASE_MODEL_PATH = stage_model(BASE_MODEL, "base", "config.json")
LORA_PATH = stage_model(LORA_REPO, "adapter", "adapter_config.json")
print("Model files are ready for vLLM.")

## 5. Grading prompt

This is the same system prompt and user-message structure used by the training notebook. Keeping the prompt format consistent is important for the adapter.

In [4]:
SYSTEM_PROMPT = (
    "You are a strict grading assistant.\n\n"
    "Your task is to evaluate whether a student's answer fulfills each grading criterion.\n\n"
    "For each criterion:\n"
    "- Output 1 if the criterion is fulfilled.\n"
    "- Output 0 if the criterion is not fulfilled.\n\n"
    "Evaluate each criterion independently.\n\n"
    "Return ONLY a valid JSON array containing 0 or 1.\n"
    "The output array must contain exactly one value for each criterion, "
    "in the same order as the criteria provided.\n\n"
    "Do not include explanations.\n"
    "Do not include markdown.\n"
    "Do not include any text before or after the JSON array."
)


def make_messages(question, desired_answer, student_answer, criteria):
    criteria_text = "\n".join(
        f"{i}. {criterion}" for i, criterion in enumerate(criteria, 1)
    )
    user_text = (
        f"Question:\n{question}\n\n"
        f"Desired answer:\n{desired_answer}\n\n"
        f"Student answer:\n{student_answer}\n\n"
        f"Criteria:\n{criteria_text}"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]

## 6. Start vLLM with the QLoRA adapter

The base model and LoRA adapter are served from local Colab copies of the files saved in Google Drive. The adapter is exposed as `model="grading"`.

Thinking is disabled server-wide because the adapter was trained with `enable_thinking=False`.

The server uses eager execution, matching `serve_qwen35_vllm.sh`, so it skips CUDA graph capture at startup. It also disables the FlashInfer sampler to avoid its optional CUDA JIT compilation. The server log is written to `/content/vllm.log`.

In [ ]:
# Stop an old server from an earlier execution of this notebook.
subprocess.run(
    ["pkill", "-f", "vllm serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(2)

vllm_log_path = "/content/vllm.log"
vllm_log = open(vllm_log_path, "w")

cmd = [
    "vllm", "serve", BASE_MODEL_PATH,
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--api-key", VLLM_API_KEY,
    "--enable-lora",
    "--lora-modules", f"{SERVED_MODEL_NAME}={LORA_PATH}",
    "--max-lora-rank", str(MAX_LORA_RANK),
    "--max-model-len", str(MAX_MODEL_LEN),
    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
    "--dtype", VLLM_DTYPE,
    "--language-model-only",
    "--default-chat-template-kwargs", '{"enable_thinking": false}',
]

if ENFORCE_EAGER:
    cmd.append("--enforce-eager")

# Match serve_qwen35_vllm.sh and avoid FlashInfer sampler JIT compilation.
server_env = os.environ.copy()
server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

if USE_BITSANDBYTES_4BIT:
    cmd += ["--quantization", "bitsandbytes"]

print("Starting vLLM...")
print(" ".join(
    "***API_KEY***" if part == VLLM_API_KEY else part
    for part in cmd
))

vllm_process = subprocess.Popen(
    cmd,
    stdout=vllm_log,
    stderr=subprocess.STDOUT,
    env=server_env,
)

headers = {"Authorization": f"Bearer {VLLM_API_KEY}"}
models_url = f"http://127.0.0.1:{PORT}/v1/models"

startup_started = time.time()
deadline = startup_started + 30 * 60
next_progress = startup_started + 60
last_status = None

while time.time() < deadline:
    if vllm_process.poll() is not None:
        vllm_log.flush()
        print(Path(vllm_log_path).read_text(errors="replace")[-12000:])
        raise RuntimeError("vLLM exited before becoming ready. See the log above.")

    try:
        response = requests.get(models_url, headers=headers, timeout=5)
        last_status = response.status_code
        if response.ok:
            break
    except requests.RequestException:
        pass

    if time.time() >= next_progress:
        vllm_log.flush()
        lines = Path(vllm_log_path).read_text(errors="replace").splitlines()
        latest = lines[-1][-300:] if lines else "No server log output yet."
        elapsed_minutes = int((time.time() - startup_started) / 60)
        print(f"Still starting after {elapsed_minutes} min. Latest log: {latest}")
        next_progress = time.time() + 60

    time.sleep(5)
else:
    vllm_log.flush()
    print(Path(vllm_log_path).read_text(errors="replace")[-12000:])
    raise TimeoutError(
        f"vLLM did not become ready within 30 minutes. Last HTTP status: {last_status}. "
        "Check the log before rerunning this cell; the server process may still be loading."
    )

print("vLLM is ready.")
print(response.json())

### If vLLM is still starting or fails

Run the cell below to check whether the existing server process has become ready and to inspect the latest log lines. A timeout in the startup cell does not stop the server process.


In [ ]:
# Use this after a startup timeout without restarting the server.
if "vllm_process" in globals():
    exit_code = vllm_process.poll()
    if exit_code is None:
        print("vLLM process is still running.")
    else:
        print(f"vLLM process exited with code {exit_code}.")

try:
    status = requests.get(models_url, headers=headers, timeout=5)
    print(f"Model endpoint HTTP status: {status.status_code}")
    if status.ok:
        print("vLLM is ready.")
except requests.RequestException as exc:
    print(f"Model endpoint is not ready yet: {exc}")

print(Path("/content/vllm.log").read_text(errors="replace")[-4000:])

## 7. Test the fine-tuned grading model inside Colab

In [ ]:
from openai import OpenAI

local_client = OpenAI(
    base_url=f"http://127.0.0.1:{PORT}/v1",
    api_key=VLLM_API_KEY,
)

criteria = [
    "States that HTTPS encrypts traffic.",
    "States that HTTPS uses TLS/SSL.",
    "States that HTTPS protects against interception or tampering.",
]

response = local_client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=make_messages(
        question="Explain why HTTPS is more secure than HTTP.",
        desired_answer=(
            "HTTPS uses TLS to encrypt traffic between the client and server, "
            "helping protect data from interception and tampering."
        ),
        student_answer="HTTPS encrypts the connection so attackers cannot easily read the traffic.",
        criteria=criteria,
    ),
    temperature=0,
    max_tokens=128,
    stream=False,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    },
)

print("Raw model output:")
print(response.choices[0].message.content)

## 8. Create a public HTTPS tunnel for your local Docker app

This starts a temporary Cloudflare Quick Tunnel from the public internet to Colab's vLLM port.

The resulting URL is **ephemeral**. Keep the URL and API key private and use this only for your demo/development setup.

In [ ]:
# Stop an old tunnel if this cell is rerun.
if "tunnel_process" in globals() and tunnel_process.poll() is None:
    tunnel_process.terminate()
    tunnel_process.wait(timeout=10)

tunnel_log_path = "/content/cloudflared.log"
tunnel_log = open(tunnel_log_path, "w")

tunnel_process = subprocess.Popen(
    [
        "/usr/local/bin/cloudflared",
        "tunnel",
        "--url", f"http://127.0.0.1:{PORT}",
        "--no-autoupdate",
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
deadline = time.time() + 90
PUBLIC_URL = None

while time.time() < deadline:
    if tunnel_process.poll() is not None:
        tunnel_log.flush()
        print(Path(tunnel_log_path).read_text(errors="replace"))
        raise RuntimeError("cloudflared exited before producing a public URL.")

    tunnel_log.flush()
    log_text = Path(tunnel_log_path).read_text(errors="replace")
    match = pattern.search(log_text)
    if match:
        PUBLIC_URL = match.group(0)
        break

    time.sleep(1)

if not PUBLIC_URL:
    print(Path(tunnel_log_path).read_text(errors="replace"))
    raise TimeoutError("Could not find the Cloudflare tunnel URL.")

PUBLIC_VLLM_BASE_URL = f"{PUBLIC_URL}/v1"

print("Public vLLM base URL:")
print(PUBLIC_VLLM_BASE_URL)
print()
print("Use these values in your local Docker app:")
print(f"VLLM_BASE_URL={PUBLIC_VLLM_BASE_URL}")
print("VLLM_API_KEY=<the VLLM_API_KEY secret used in this notebook>")
print(f"VLLM_MODEL={SERVED_MODEL_NAME}")

## 9. Verify the public tunnel from Colab

In [ ]:
public_client = OpenAI(
    base_url=PUBLIC_VLLM_BASE_URL,
    api_key=VLLM_API_KEY,
)

models = public_client.models.list()
print("Public endpoint is reachable.")
print([model.id for model in models.data])

## 10. Local Docker application example

Your local Docker container only needs normal outbound HTTPS access. It does **not** need to be on the same network as Colab.

Set environment variables in your local project:

```env
VLLM_BASE_URL=https://YOUR-RANDOM-SUBDOMAIN.trycloudflare.com/v1
VLLM_API_KEY=YOUR_SECRET
VLLM_MODEL=grading
```

Then use the OpenAI Python client:

```python
import os
from openai import OpenAI

client = OpenAI(
    base_url=os.environ["VLLM_BASE_URL"],
    api_key=os.environ["VLLM_API_KEY"],
)

response = client.chat.completions.create(
    model=os.environ.get("VLLM_MODEL", "grading"),
    messages=[
        {"role": "system", "content": "Your system prompt here"},
        {"role": "user", "content": "Your grading request here"},
    ],
    temperature=0,
    stream=False,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    },
)

print(response.choices[0].message.content)
```

For the real grading flow, send messages using the same `SYSTEM_PROMPT` and `make_messages(...)` format defined above.

### Important

- Keep the Colab runtime running while your local application uses the endpoint.
- The Quick Tunnel URL changes after a restart.
- Use `stream=False` for this demo endpoint.
- This setup is appropriate for a demo, not a production deployment.

## 11. Stop the server and tunnel when finished

In [ ]:
if "tunnel_process" in globals() and tunnel_process.poll() is None:
    tunnel_process.terminate()
    print("Stopped Cloudflare tunnel.")

if "vllm_process" in globals() and vllm_process.poll() is None:
    vllm_process.terminate()
    print("Stopped vLLM server.")